# Análisis preliminar de series — Visitantes internacionales a Guatemala

Construcción de las series mensuales requeridas y partición cronológica de entrenamiento (70%) y prueba (30%). La lógica de partición reutiliza el criterio aplicado en `laboratorio1.ipynb`: al ser una serie de tiempo, la división respeta el orden temporal (sin mezclar meses) para evitar que información futura entre al entrenamiento.

**Series a construir:**
- **a. Obligatoria:** Total mensual de viajeros internacionales.
- **b. Dos categorías seleccionadas** (de un total de 5 posibles): **Vías de ingreso** y **Regiones geográficas** (`Región dos`, Top 3). La justificación de esta elección se detalla en la sección 4.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.dpi": 110
})

In [ ]:
CANDIDATOS_DATOS = [
    Path("../data/raw/Base_Migracion_2009-2026jun.xlsx"),
    Path("data/raw/Base_Migracion_2009-2026jun.xlsx"),
    Path("Base_Migracion_2009-2026jun.xlsx"),
]
RUTA_DATOS = next((p for p in CANDIDATOS_DATOS if p.exists()), None)
assert RUTA_DATOS is not None, (
    "No se encontró Base_Migracion_2009-2026jun.xlsx en las rutas candidatas"
)
RUTA_DATOS

## 1. Carga y preparación

Se construye la fecha mensual a partir de año y mes, y se conserva únicamente `Turista` y `Excursionista` como categorías comparables en todo el período (ver nota de quiebre metodológico 2022→2023 en el codebook: `Viajero` se reclasifica y `Cruceristas` deja de registrarse desde 2023). Esta base filtrada (`visitantes`) es la que se usa tanto para construir las series como para la partición.

In [ ]:
datos_raw = pd.read_excel(RUTA_DATOS, sheet_name="Datos")

datos = datos_raw.copy()
datos["Fecha"] = pd.to_datetime(
    dict(year=datos["Año"], month=datos["Mes cod"], day=1),
    errors="coerce"
)
datos = datos.sort_values("Fecha").reset_index(drop=True)

tipos_comparables = ["Turista", "Excursionista"]
visitantes = datos.loc[datos["Tipo de Viajero"].isin(tipos_comparables)].copy()

meses_esperados = pd.date_range(datos["Fecha"].min(), datos["Fecha"].max(), freq="MS")

print(f"Filas totales: {len(datos):,} | Filas comparables: {len(visitantes):,}")
print(
    f"Periodo: {datos['Fecha'].min():%Y-%m} a {datos['Fecha'].max():%Y-%m} "
    f"({len(meses_esperados)} meses)"
)

## 2. Serie obligatoria — Total mensual

Se agregan todos los registros comparables por mes para obtener la serie total de visitantes internacionales.

In [ ]:
serie_total = (
    visitantes.groupby("Fecha")["Viajero"]
              .sum()
              .reindex(meses_esperados, fill_value=0)
              .rename("Total")
)

fig, ax = plt.subplots()
ax.plot(serie_total, color="#222222", lw=1.5)
ax.set(title="Serie obligatoria — Total mensual de visitantes", xlabel="Fecha", ylabel="Visitantes")
plt.tight_layout()
plt.show()

serie_total.describe().to_frame("Total")

## 3. División cronológica de entrenamiento y prueba (70/30)

Misma lógica que `laboratorio1.ipynb`: se ordenan los meses únicos, se toma el 70% inicial como entrenamiento y el 30% final como prueba, y todas las filas de un mismo mes quedan en el mismo conjunto. Al ser una serie de tiempo, la partición no puede ser aleatoria: mezclar meses futuros en el entrenamiento produciría fuga de información y una evaluación optimista de cualquier modelo.

Esta misma partición de fechas (`fechas_train` / `fechas_test`) se reutiliza más abajo para todas las series de las categorías seleccionadas, de modo que todas comparten exactamente el mismo corte temporal.

In [ ]:
fechas = pd.DatetimeIndex(sorted(visitantes["Fecha"].unique()))
n_train = int(np.floor(0.70 * len(fechas)))
fechas_train = fechas[:n_train]
fechas_test = fechas[n_train:]
fecha_corte = fechas_train[-1]

train = visitantes.loc[visitantes["Fecha"].isin(fechas_train)].copy()
test = visitantes.loc[visitantes["Fecha"].isin(fechas_test)].copy()

indice_train = pd.date_range(fechas_train.min(), fechas_train.max(), freq="MS")
indice_test = pd.date_range(fechas_test.min(), fechas_test.max(), freq="MS")

particion = pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Prueba"],
    "Inicio": [fechas_train.min(), fechas_test.min()],
    "Fin": [fechas_train.max(), fechas_test.max()],
    "Meses": [len(fechas_train), len(fechas_test)],
    "Porcentaje de meses": [
        100 * len(fechas_train) / len(fechas),
        100 * len(fechas_test) / len(fechas)
    ],
    "Filas": [len(train), len(test)]
})
display(particion)

serie_total_train = serie_total.reindex(indice_train)
serie_total_test = serie_total.reindex(indice_test)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(serie_total, color="#4C78A8", lw=1.5)
ax.axvspan(fechas_train.min(), fechas_train.max(), color="#59A14F",
           alpha=0.12, label="Entrenamiento")
ax.axvspan(fechas_test.min(), fechas_test.max(), color="#E15759",
           alpha=0.12, label="Prueba")
ax.axvline(fecha_corte, color="black", ls="--", lw=1)
ax.set(title="Partición temporal de la serie total", xlabel="Fecha", ylabel="Visitantes")
ax.legend()
plt.tight_layout()
plt.show()

La prueba comienza inmediatamente después del último mes de entrenamiento (`fecha_corte`), sin traslape. Con 210 meses disponibles, el 70% corresponde a 147 meses de entrenamiento (2009-01 a 2021-03) y el 30% restante a 63 meses de prueba (2021-04 a 2026-06); esa ventana de prueba incluye buena parte del período de recuperación pos-pandemia, lo que la vuelve una evaluación exigente para cualquier modelo.

## 4. Categorías seleccionadas: Vías de ingreso y Regiones geográficas

El enunciado permite elegir 2 de 5 categorías: países de residencia, regiones geográficas, vías de ingreso, fronteras o tipo de viajero. Se eligieron **Vías de ingreso** y **Regiones geográficas (`Región dos`)** porque son las dos categorías cuya serie no arrastra ningún quiebre metodológico dentro del período 2009–2026:

- **Países de residencia queda descartada**: el segundo país con mayor acumulado histórico es *Guatemala* (13.9M), pero esa etiqueta **desaparece por completo desde 2023** por el cambio de granularidad de la columna `País` (de país individual a agrupación de mercado). El Top 3 por acumulado incluiría una serie que cae a cero de forma abrupta y permanente, no por una caída real de viajeros sino por un artefacto de la fuente.
- **Tipo de viajero queda descartada**: `Cruceristas` deja de reportarse desde 2023 y `Viajero` sufre una reclasificación que reduce su nivel en ~70% en 2023 (de ~1.06M a ~0.33M), ambos quiebres de medición y no de comportamiento real.
- **Fronteras** también resultó viable (las tres principales — La Aurora, Valle Nuevo, San Cristóbal — están presentes los 18 años), pero se priorizaron Vías y Regiones por dar una lectura más agregada y directamente interpretable del comportamiento del turismo (modo de ingreso vs. origen geográfico), en lugar de puntos de control específicos.

El criterio de selección del Top 3 (regiones) usa el acumulado de **todo el período de estudio**, tal como pide el enunciado, no un año específico.

### 4a. Vías de ingreso

Las tres vías (`Aérea`, `Terrestre`, `Marítima`) cubren el 100% de los registros por definición, así que no requieren un criterio de Top-N.

In [ ]:
vias_requeridas = ["Aérea", "Terrestre", "Marítima"]

series_via = (
    visitantes.pivot_table(
        index="Fecha", columns="Vía", values="Viajero",
        aggfunc="sum", fill_value=0
    )
    .reindex(index=meses_esperados, columns=vias_requeridas, fill_value=0)
)
series_via_train = series_via.reindex(indice_train)
series_via_test = series_via.reindex(indice_test)

fig, ax = plt.subplots()
series_via.plot(ax=ax, lw=1.2)
ax.set(title="Series mensuales por vía de ingreso", xlabel="Fecha", ylabel="Visitantes")
plt.tight_layout()
plt.show()

### 4b. Regiones geográficas (`Región dos`, Top 3)

El Top 3 se calcula sobre el acumulado de `visitantes` en todo el período (2009–2026), no sobre el entrenamiento ni sobre un año particular.

In [ ]:
top3_regiones = (
    visitantes.groupby("Región dos")["Viajero"]
              .sum()
              .nlargest(3)
              .index
              .tolist()
)
print("Top 3 de regiones (acumulado 2009-2026):", ", ".join(top3_regiones))

series_region = (
    visitantes.loc[visitantes["Región dos"].isin(top3_regiones)]
              .pivot_table(
                  index="Fecha", columns="Región dos", values="Viajero",
                  aggfunc="sum", fill_value=0
              )
              .reindex(index=meses_esperados, columns=top3_regiones, fill_value=0)
)
series_region_train = series_region.reindex(indice_train)
series_region_test = series_region.reindex(indice_test)

fig, ax = plt.subplots()
series_region.plot(ax=ax, lw=1.2)
ax.set(title="Series mensuales por región geográfica (Top 3)", xlabel="Fecha", ylabel="Visitantes")
plt.tight_layout()
plt.show()

## 5. Resumen: series de entrenamiento y prueba

Total de series construidas: 1 obligatoria (Total) + 3 (vías) + 3 (regiones) = **7 series**, todas particionadas con el mismo corte cronológico (`fecha_corte`).

In [ ]:
todas_las_series = {"Total": serie_total}
todas_las_series.update({f"Vía — {c}": series_via[c] for c in series_via.columns})
todas_las_series.update({f"Región — {c}": series_region[c] for c in series_region.columns})

metadatos_series = pd.DataFrame([
    {
        "Serie": nombre,
        "Inicio": serie.index.min(),
        "Fin": serie.index.max(),
        "Observaciones": len(serie),
        "Meses train": len(indice_train),
        "Meses test": len(indice_test),
        "Total acumulado": serie.sum()
    }
    for nombre, serie in todas_las_series.items()
])
display(metadatos_series)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
axes = axes.ravel()

paneles = [("Total", serie_total_train, serie_total_test)]
paneles += [(f"Vía — {c}", series_via_train[c], series_via_test[c]) for c in series_via.columns]
paneles += [(f"Región — {c}", series_region_train[c], series_region_test[c]) for c in series_region.columns]

for ax, (nombre, serie_tr, serie_te) in zip(axes, paneles):
    ax.plot(serie_tr, color="#4C78A8", lw=1.3, label="Entrenamiento")
    ax.plot(serie_te, color="#E15759", lw=1.3, label="Prueba")
    ax.set_title(nombre, fontsize=10)
    ax.legend(fontsize=7)

for ax in axes[len(paneles):]:
    ax.axis("off")

fig.suptitle("Las 7 series — entrenamiento vs. prueba", y=1.01)
plt.tight_layout()
plt.show()

## Próximos pasos

Con las 7 series (Total, 3 vías, 3 regiones) definidas en orden cronológico y ya particionadas, los siguientes pasos del análisis preliminar son evaluar estacionariedad (ADF/KPSS), revisar ACF/PACF y decidir transformaciones (log, diferenciación estacional) por serie antes de ajustar modelos — siguiendo el mismo enfoque que la sección 11 de `laboratorio1.ipynb`.